# Langchain RAG

A typical RAG application has two main components:

**Indexing**: a pipeline for ingesting data from a source and indexing it. This usually happens offline.

**Retrieval and generation**: the actual RAG chain, which takes the user query at run time and retrieves the relevant data from the index, then passes that to the model.

The most common full sequence from raw data to answer looks like:

### Indexing
1. **Load**: First we need to load our data. This is done in langchain with Document Loaders classes.
2. **Split**: Text splitters break large Documents into smaller chunks. This is necessary because embedding models have a finite context window.
3. **Embed**: Then we need to convert those chunks into vectors. This is done with an embedding model.
4. **Store**: We need somewhere to store and index our vectors from the text chunks, so that we can search over them later. This is done using a VectorStore.

### Retrieval and generation
5. **Retrieve**: Given a user input, relevant splits are retrieved from storage using a Retriever.
6. **Generate**: A LLM produces an answer using a prompt that includes both the question and the retrieved data

![Indexing pipeline](./indexing-pipeline.jpg)


In [1]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

Enter API key for OpenAI: ··········


In [2]:

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

import bs4

# 1) LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.environ["OPENAI_API_KEY"]
)

In [3]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [4]:
# 2) Embeddings + Vectorstore
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_store = InMemoryVectorStore(embeddings)

In this guide we’ll build an app that answers questions about the website's content. The specific website we will use is the [LLM Powered Autonomous Agents](https://lilianweng.github.io/posts/2023-06-23-agent/) blog post by Lilian Weng, which allows us to ask questions about the contents of the post.

We can create a simple indexing pipeline and RAG chain to do this in ~50 lines of code.

 Load & Split Your Documents

In [5]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
# 3) Load + split documents
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)

Embed & Store Vectors

In [7]:
# 4) Index
_ = vector_store.add_documents(all_splits)


In [8]:
retriever = vector_store.as_retriever(
    search_type = "similarity",     # "mmr" or "similarity_score_threshold" also work
    search_kwargs = {"k": 4}
)


In [9]:
# 5) Prompt
SYSTEM = """You are an expert assistant.
Answer *only* from the context between <context></context>;
if the answer isn’t there, say “I don't know.”"""

USER = """<context>
{context}
</context>

Question: {input}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("user", USER)
])

In [10]:

from langchain_core.runnables import RunnableMap, RunnablePassthrough

# 6) Combine documents
def combine_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 7) RAG chain (modern LCEL)
rag_chain = (
    RunnableMap({
        "input": RunnablePassthrough()
    })
    | RunnableMap({
        "docs": lambda x: retriever.invoke(x["input"]),
        "input": lambda x: x["input"]
    })
    | RunnableMap({
        "context": lambda x: combine_docs(x["docs"]),
        "docs": lambda x: x["docs"],
        "input": lambda x: x["input"]
    })
    | prompt
    | llm
    | (lambda output, inputs: {
        "answer": output.content,
        "context": inputs["context"]
    })
)

In [13]:
from langchain_core.runnables import RunnablePassthrough

# 6) Combine documents
def combine_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 7) RAG chain (modern LCEL)
rag_chain = (
    {
        "input": RunnablePassthrough(),
        "context": retriever | combine_docs
    }
    | prompt
    | llm
)

# 8) Query

question = "What is Task Decomposition?"
raw = rag_chain.invoke(question)
result = {
    "answer": raw.content,
    "context": combine_docs(retriever.invoke(question))
}

print("Answer:")
print(result["answer"])
print("\nContext:")
print(result["context"])

Answer:
Task Decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.

Context:
Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Another quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external classical planner to do long-horizon planning. This approach utilizes the Planning Domain Definition Language (PDDL) as an intermediate interface to describe the planning problem. In this process, LLM (1) translates the problem into “Problem PDDL”, then (2) requests a classical planner to generate a PDDL plan based on an existing “Domain PDDL”, and finally (3) translates the PDDL pla